# Phase 2 — Baseline Supervised Model (EF regression + HFrEF classification)

Goal: a **working, honest baseline** before adding semi-supervised pretraining, quality-awareness, or uncertainty. This establishes the pipeline (data loading, training loop, checkpointing, metrics) end-to-end on the small dev subsample first.

**Design choices for free-Colab (Section 28.4 of the roadmap):**
- Single **middle frame** per video, not the full video — a lightweight 2D CNN, not 3D. This is intentionally simple; a temporal model is a later ablation, not the starting point.
- Small ResNet18 (ImageNet-pretrained) — fast to train, low memory.
- Trained and validated on the **~180-video dev subsample** first. Full-dataset training is a separate, later run once this pipeline is verified.
- Checkpoints saved every epoch to Drive using the helpers from notebook 00.

**Two outputs from one model:**
1. EF regression (continuous, %), evaluated with MAE / R².
2. HFrEF classification (EF < 40), evaluated with AUC / F1 — since Notebook 00's EDA showed only ~12.6% positive rate, we use a class-weighted loss term.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, json, time
import numpy as np
import pandas as pd

PROJECT_ROOT = '/content/drive/MyDrive/echonet_project'
DATA_DIR = os.path.join(PROJECT_ROOT, 'data/raw', 'EchoNet-Dynamic', 'EchoNet-Dynamic')
VIDEOS_DIR = os.path.join(DATA_DIR, 'Videos')
SUBSAMPLE_CSV = os.path.join(PROJECT_ROOT, 'data/subsample', 'subsample_filelist.csv')

assert os.path.exists(SUBSAMPLE_CSV), 'Run 00_setup_and_eda.ipynb first to build the subsample.'
assert os.path.isdir(VIDEOS_DIR), f'Videos folder not found at {VIDEOS_DIR}'

sub_df = pd.read_csv(SUBSAMPLE_CSV)
print('Subsample:', sub_df.shape)
sub_df.head()

In [ ]:
!pip install --quiet opencv-python-headless
import cv2
print('opencv ready')

## Extract the middle frame from each subsample video

Caches frames as .npy on Drive so this only needs to run once — re-running later training cells won't re-decode video.

In [ ]:
FRAME_CACHE_DIR = os.path.join(PROJECT_ROOT, 'data/subsample', 'frames')
os.makedirs(FRAME_CACHE_DIR, exist_ok=True)

def get_middle_frame(video_path):
    cap = cv2.VideoCapture(video_path)
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    mid = max(0, n_frames // 2)
    cap.set(cv2.CAP_PROP_POS_FRAMES, mid)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        return None
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    return frame

filename_col = 'FileName' if 'FileName' in sub_df.columns else [c for c in sub_df.columns if 'file' in c.lower() and 'name' in c.lower()][0]

cached, missing = 0, 0
for _, row in sub_df.iterrows():
    fname = str(row[filename_col])
    if not fname.endswith('.avi'):
        fname = fname + '.avi'
    cache_path = os.path.join(FRAME_CACHE_DIR, fname.replace('.avi', '.npy'))
    if os.path.exists(cache_path):
        cached += 1
        continue
    video_path = os.path.join(VIDEOS_DIR, fname)
    if not os.path.exists(video_path):
        missing += 1
        continue
    frame = get_middle_frame(video_path)
    if frame is not None:
        np.save(cache_path, frame)
        cached += 1
    else:
        missing += 1

print(f'Frames cached: {cached}, missing/failed: {missing}')

## Dataset + DataLoader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

ef_col = 'EF' if 'EF' in sub_df.columns else [c for c in sub_df.columns if 'ef' in c.lower()][0]
HFrEF_THRESHOLD = 40

class EchoFrameDataset(Dataset):
    def __init__(self, df, frame_dir, filename_col, ef_col, train=True):
        self.df = df.reset_index(drop=True)
        self.frame_dir = frame_dir
        self.filename_col = filename_col
        self.ef_col = ef_col
        mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]  # ImageNet stats, since we use a pretrained backbone
        aug = [T.RandomHorizontalFlip()] if train else []
        self.transform = T.Compose([
            T.ToPILImage(), T.Resize((112, 112)), *aug, T.ToTensor(), T.Normalize(mean, std)
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = str(row[self.filename_col])
        if not fname.endswith('.avi'):
            fname = fname + '.avi'
        cache_path = os.path.join(self.frame_dir, fname.replace('.avi', '.npy'))
        frame = np.load(cache_path)
        img = self.transform(frame)
        ef = float(row[self.ef_col])
        label = 1.0 if ef < HFrEF_THRESHOLD else 0.0
        return img, torch.tensor(ef, dtype=torch.float32), torch.tensor(label, dtype=torch.float32)

# Drop any rows whose frame extraction failed
valid_mask = sub_df[filename_col].apply(lambda f: os.path.exists(
    os.path.join(FRAME_CACHE_DIR, (str(f) if str(f).endswith('.avi') else str(f)+'.avi').replace('.avi', '.npy'))
))
clean_df = sub_df[valid_mask].reset_index(drop=True)
print(f'Usable videos: {len(clean_df)} / {len(sub_df)}')

# Simple 80/20 split within the dev subsample (full train/val/test split is used later at full scale)
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(clean_df, test_size=0.2, random_state=42,
                                     stratify=(clean_df[ef_col] < HFrEF_THRESHOLD))

train_ds = EchoFrameDataset(train_df, FRAME_CACHE_DIR, filename_col, ef_col, train=True)
val_ds = EchoFrameDataset(val_df, FRAME_CACHE_DIR, filename_col, ef_col, train=False)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2)
print('Train:', len(train_ds), 'Val:', len(val_ds))

In [ ]:
# Reproducibility: fix random seeds (used for all runs from this version onward)
import random
import numpy as np
import torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
# Note: GPU kernels (cuDNN) can still be non-deterministic, so re-runs may differ slightly in the last digits.

## Model: ResNet18 backbone, two heads (EF regression + HFrEF logit)

In [ ]:
import torch.nn as nn
import torchvision.models as models

class EchoBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        n_feat = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.ef_head = nn.Linear(n_feat, 1)
        self.cls_head = nn.Linear(n_feat, 1)

    def forward(self, x):
        feat = self.backbone(x)
        ef_pred = self.ef_head(feat).squeeze(-1)
        cls_logit = self.cls_head(feat).squeeze(-1)
        return ef_pred, cls_logit

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cpu':
    print('WARNING: no GPU detected. Go to Runtime > Change runtime type > GPU (T4), then re-run this notebook from the top.')

model = EchoBaseline().to(device)

# Initialize the EF regression head's bias to the dataset mean EF, so the model starts near a
# sensible prediction instead of near 0 — this alone removes most of the early, uninformative
# high-MAE epochs and lets training focus on learning real signal from the images.
with torch.no_grad():
    mean_ef = float(clean_df[ef_col].mean())
    model.ef_head.bias.fill_(mean_ef)
    model.ef_head.weight.zero_()
print(f'EF head bias initialized to dataset mean: {mean_ef:.2f}')

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# Class-imbalance-aware weight for the classification loss, from the ~12.6% positive rate found in the EDA
pos_rate = (clean_df[ef_col] < HFrEF_THRESHOLD).mean()
pos_weight = torch.tensor([(1 - pos_rate) / max(pos_rate, 1e-6)], device=device)
print(f'HFrEF positive rate: {pos_rate:.3f} -> pos_weight: {pos_weight.item():.2f}')

cls_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
reg_criterion = nn.L1Loss()  # MAE loss, robust and directly interpretable in EF %

## Checkpoint helpers (same pattern as notebook 00 — copy kept local so this notebook is self-contained)

In [ ]:
EXP_NAME = 'baseline_resnet18_subsample_v2'  # new name -- v1 checkpoints used a worse init, don’t resume from those

def save_checkpoint(model, optimizer, epoch, metrics, exp_name):
    ckpt_dir = os.path.join(PROJECT_ROOT, 'checkpoints', exp_name)
    os.makedirs(ckpt_dir, exist_ok=True)
    path = os.path.join(ckpt_dir, f'epoch_{epoch:03d}.pt')
    torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(), 'metrics': metrics}, path)
    log_path = os.path.join(PROJECT_ROOT, 'logs', f'{exp_name}.jsonl')
    with open(log_path, 'a') as f:
        f.write(json.dumps({'epoch': epoch, 'time': time.time(), **metrics}) + '\n')
    print(f'[checkpoint] saved epoch {epoch} -> {path}')

def load_latest_checkpoint(model, optimizer, exp_name):
    ckpt_dir = os.path.join(PROJECT_ROOT, 'checkpoints', exp_name)
    ckpts = sorted(glob.glob(os.path.join(ckpt_dir, 'epoch_*.pt')))
    if not ckpts:
        print('[checkpoint] none found, starting from scratch')
        return 0
    state = torch.load(ckpts[-1], map_location=device)
    model.load_state_dict(state['model_state'])
    optimizer.load_state_dict(state['optimizer_state'])
    print(f'[checkpoint] resumed from {ckpts[-1]} (epoch {state["epoch"]})')
    return state['epoch'] + 1

start_epoch = load_latest_checkpoint(model, optimizer, EXP_NAME)

## Train

Small number of epochs on purpose — this is a smoke test to verify the pipeline, not the final result. Scale up (more epochs, full dataset) only after this runs cleanly.

In [ ]:
from sklearn.metrics import roc_auc_score, f1_score, mean_absolute_error

N_EPOCHS = 15  # bumped up now that the bias-init trick gives faster, more meaningful convergence per epoch
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    all_ef_true, all_ef_pred, all_cls_true, all_cls_prob = [], [], [], []
    for imgs, ef_true, cls_true in loader:
        imgs, ef_true, cls_true = imgs.to(device), ef_true.to(device), cls_true.to(device)
        with torch.set_grad_enabled(train):
            ef_pred, cls_logit = model(imgs)
            loss = reg_criterion(ef_pred, ef_true) + cls_criterion(cls_logit, cls_true)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        all_ef_true.extend(ef_true.detach().cpu().numpy())
        all_ef_pred.extend(ef_pred.detach().cpu().numpy())
        all_cls_true.extend(cls_true.detach().cpu().numpy())
        all_cls_prob.extend(torch.sigmoid(cls_logit).detach().cpu().numpy())
    n = len(loader.dataset)
    mae = mean_absolute_error(all_ef_true, all_ef_pred)
    try:
        auc = roc_auc_score(all_cls_true, all_cls_prob)
    except ValueError:
        auc = float('nan')  # can happen if a batch/split has only one class present
    f1 = f1_score(all_cls_true, [1 if p > 0.5 else 0 for p in all_cls_prob], zero_division=0)
    return {'loss': total_loss / n, 'mae': mae, 'auc': auc, 'f1': f1}

for epoch in range(start_epoch, start_epoch + N_EPOCHS):
    train_metrics = run_epoch(train_loader, train=True)
    val_metrics = run_epoch(val_loader, train=False)
    metrics = {f'train_{k}': v for k, v in train_metrics.items()}
    metrics.update({f'val_{k}': v for k, v in val_metrics.items()})
    print(f"Epoch {epoch}: train_mae={train_metrics['mae']:.2f} val_mae={val_metrics['mae']:.2f} "
          f"val_auc={val_metrics['auc']:.3f} val_f1={val_metrics['f1']:.3f}")
    save_checkpoint(model, optimizer, epoch, metrics, EXP_NAME)

## Sanity read of the training log

In [ ]:
log_path = os.path.join(PROJECT_ROOT, 'logs', f'{EXP_NAME}.jsonl')
log_df = pd.read_json(log_path, lines=True)
log_df[['epoch', 'train_mae', 'val_mae', 'val_auc', 'val_f1']]

## What to check before moving on

- `val_mae` should trend down across epochs (even if noisy — 144 training videos is small).
- `val_auc` for HFrEF should be meaningfully above 0.5 by the last epoch (if it stays near 0.5 or is NaN often, note it — may just reflect subsample size, not a real problem).
- Do **not** treat these numbers as paper-worthy results — this is a pipeline smoke test on 180 videos. The next step is scaling to the full dataset only after this looks structurally sound (loss decreasing, no crashes, checkpoints resuming correctly).
- Once confirmed, the next notebook adds the quality-awareness and uncertainty/abstention components described in the roadmap, still starting on the subsample before any full-scale run.